## Indexing collections

Query performance can be improved by creating indexes over tables of the collection. 
By default, the index creation is minimal to enable faster insertion of data into the collection. 
However, after a part of a collection becomes complete (e.g. a collection table or a detached layer table is filled with data), you should also create corresponding index to enable efficient querying over the collection.


### Creating collection index (for attached layers)

...

### Creating layer index (for detached layer)

...

### Creating layer n-gram index (for detached layer)

N-gram index enables to search for n-grams of layer attributes. 
For example, if we create a bigram index on an attribute with values `['see', 'on', 'esimene', 'lause']`, then we can search for two-word phrases, such as *'see-on'*, *'on-esimene'*, *'esimene-lause'*.

Example. First, create a new collection:

In [1]:
from estnltk import Text
from estnltk.storage.postgres import PostgresStorage, delete_schema
from estnltk.taggers import VabamorfTagger

In [2]:
storage = PostgresStorage(dbname='test_db', pgpass_file='~/.pgpass',
                          schema='my_schema', create_schema_if_missing=True)

collection = storage.add_collection('collection_with_layers')

with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('See ei ole midagi.').tag_layer("sentences"))

INFO:storage.py:73: connecting to host: 'localhost', port: '5432', dbname: 'test_db', user: 'postgres'
INFO:storage.py:94: new schema 'my_schema' created
INFO:storage.py:124: schema: 'my_schema', temporary: False, role: 'postgres'
INFO:storage.py:260: new empty collection 'collection_with_layers' created
INFO:collection_text_object_inserter.py:104: inserted 3 texts into the collection 'collection_with_layers'


To build an ngram index, provide an argument `ngram_index` when creating a new layer.
The following code creates a bigram index on an attribute *lemma* for a newly created layer *indexed_layer*:

In [3]:
indexed_layer = 'indexed_layer'
tagger = VabamorfTagger(disambiguate=False, output_layer=indexed_layer)

collection.create_layer(tagger=tagger, ngram_index={"lemma": 2})

INFO:collection.py:1331: collection: 'collection_with_layers'
INFO:collection.py:1350: preparing to create a new layer: 'indexed_layer'
INFO:collection.py:1032: detached layer 'indexed_layer' created from template
INFO:collection.py:1380: inserting data into the 'indexed_layer' layer table
INFO:collection_detached_layer_inserter.py:87: inserted 3 detached 'indexed_layer' layers into the collection 'collection_with_layers'
INFO:collection.py:1421: layer created: 'indexed_layer'


To search an ngram index, use `LayerNgramQuery` query:

Search entries containing lemma bigram 'see-olema':

In [4]:
from estnltk.storage.postgres import LayerNgramQuery

q = LayerNgramQuery( { indexed_layer: {
        "lemma": [("see", "olema")]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')
1 Text(text='See on teine lause.')


Search 'teine-lause' OR 'olema-esimene':

In [5]:
q = LayerNgramQuery( { indexed_layer: {
        "lemma":  [("teine", "lause"), ("olema", "esimene")]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')
1 Text(text='See on teine lause.')


Search 'see-olema' AND 'olema-esimene':

In [6]:
q = LayerNgramQuery( { indexed_layer: {
        "lemma":  [[("see", "olema"), ("olema", "esimene")]]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')


<p>
<div class="alert alert-block alert-warning">
<h4><i>Limitations of ngram indexing</i></h4> 
<p>Currently, ngram indexing cannot be applied on creating relation layers.</p>
</div>
</p>

In [7]:
storage.delete_collection( collection.name )

In [8]:
delete_schema(storage)
storage.close()

---

## Technical notes

### PostgresStorage's new indexing behaviour ( v1.7.5+ ) 

* Collection tables are indexed in the following way:
   * Once a new collection table is created via `storage.add_collection(...)`:
        * no indexes are created by default;
        * optionally, if `create_index` is set, then 2 indexes are created over its attached layers:
           * `'idx_{collection_name}_layer_data'` -- [GIN index](https://www.postgresql.org/docs/current/gin.html) over attached span layers (`(data->'layers') jsonb_path_ops`);
           * `'idx_{collection_name}_relation_layer_data'` -- GIN index over attached relation layers (`(data->'relation_layers') jsonb_path_ops`);
    * Alternatively, the attached layer indexes can be created later via `collection.create_index()` method and, `collection.drop_index()` can be used to remove the indexes;
* Detached layer tables are indexed in the following way:
   * Once a new detached layer is created via `collection.add_layer(...)`:
        * an index over `text_id` is always created:
            * `'idx_{detached_layer_table_name}__text_id'` -- index over `text_id` column;
        * optionally, if `create_index` is set, then an index over spans or relations is created:
            * `'idx_{detached_layer_table_name}_spans'` -- GIN index over layer's spans (`(data->'spans') jsonb_path_ops`) (if the layer is span layer);
            * `'idx_{detached_layer_table_name}_relations'` -- GIN index over layer's relations (`(data->'relations') jsonb_path_ops`) (if the layer is relation layer);
        * optionally, if `create_ngram_index` and `ngram_index` are set, then an array index is created:
            * `'idx_{detached_layer_table_name}_{ngram_column_name}'` -- GIN index over n-gram array column;
    * Alternatively, you can use `collection.create_layer_index(...)` method to add detached layer indexes later:
        * `collection.create_layer_index(layer_name, index_type='data')` -- creates an index over layer's spans or relations;
        * `collection.create_layer_index(layer_name, index_type='ngram_index', ngram_index={'ngram_column_name': n})` -- creates an index over n-gram data;

### PostgresStorage's old indexing behaviour (v1.7.4) 

* Collection tables are indexed in the following way:
   * Once a new collection table is created via `storage.add_collection`, 2 indexes are created over its attached layers:
        * `'idx_{collection_name}_layer_data'` -- [GIN index](https://www.postgresql.org/docs/current/gin.html) over attached span layers (`(data->'layers') jsonb_path_ops`);
        * `'idx_{collection_name}_relation_layer_data'` -- GIN index over attached relation layers (`(data->'relation_layers') jsonb_path_ops`);
   * Alternatively, the same attached layer indexes can be created via `collection.create_index()` method and, `collection.drop_index()` can be used to remove the indexes;
* Detached layer tables are indexed in the following way:
   * Once a new detached layer is created via `collection.add_layer()`:
        * an index over `text_id` is always created:
            * `'idx_{detached_layer_table_name}__text_id'` -- index over `text_id` column;
        * optionally, if `create_index` is set, then an index over spans or relations is created:
            * `'idx_{detached_layer_table_name}_spans'` -- GIN index over layer's spans (`(data->'spans') jsonb_path_ops`) (if the layer is span layer);
            * `'idx_{detached_layer_table_name}_relations'` -- GIN index over layer's relations (`(data->'relations') jsonb_path_ops`) (if the layer is relation layer);
        * optionally, if `ngram_index` is set, then an array index is created:
            * `'idx_{detached_layer_table_name}_{ngram_column_name}'` -- GIN index over n-gram array column;